In [4]:
# ============================================================
# WHATSAPP CHAT ANALYSIS - COMPLETE CODE IN ONE CELL
# ============================================================

# -------------------- IMPORT LIBRARIES --------------------

import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
import emoji


# -------------------- READ WHATSAPP CHAT --------------------

# Change this filename to your WhatsApp exported chat file
with open('Chat.txt', 'r', encoding='utf-8') as file:
    data = file.read()

print("Chat file loaded successfully!")


# -------------------- EXTRACT MESSAGES --------------------

# Regular expression to identify date and time
pattern = r'(\d{1,2}/\d{1,2}/\d{2,4}),\s(\d{1,2}:\d{2})\s-\s'

# Find positions where each message starts
matches = list(re.finditer(pattern, data))

positions = [match.start() for match in matches]

messages = []

# Split complete chat into individual messages
for i in range(len(positions)):

    start = positions[i]

    if i < len(positions) - 1:
        end = positions[i + 1]
    else:
        end = len(data)

    messages.append(data[start:end])


# -------------------- CREATE DATAFRAME --------------------

dates = []
times = []
message_text = []

# Extract date, time and message
for message in messages:

    match = re.match(pattern, message)

    if match:

        date = match.group(1)
        time = match.group(2)

        text = message[match.end():].strip()

        dates.append(date)
        times.append(time)
        message_text.append(text)


# Create DataFrame
df = pd.DataFrame({
    'date': dates,
    'time': times,
    'message': message_text
})


# -------------------- SEPARATE USER AND MESSAGE --------------------

# Split the sender and message
df[['user', 'message']] = df['message'].str.split(
    ': ',
    n=1,
    expand=True
)

# System messages don't have a sender
df['user'] = df['user'].fillna('group_notification')

df['message'] = df['message'].fillna('')


# -------------------- CONVERT DATE --------------------

# Convert date into datetime
df['date'] = pd.to_datetime(
    df['date'],
    dayfirst=True,
    errors='coerce'
)

# Remove invalid dates
df = df.dropna(subset=['date'])


# -------------------- CREATE EXTRA COLUMNS --------------------

# Year
df['year'] = df['date'].dt.year

# Month number
df['month'] = df['date'].dt.month

# Month name
df['month_name'] = df['date'].dt.month_name()

# Day
df['day'] = df['date'].dt.day

# Day name
df['day_name'] = df['date'].dt.day_name()

# Hour
df['hour'] = pd.to_datetime(
    df['time'],
    format='%H:%M',
    errors='coerce'
).dt.hour

# Month-Year
df['month_year'] = df['date'].dt.to_period('M').astype(str)


# -------------------- BASIC STATISTICS --------------------

# Total messages
total_messages = df.shape[0]

# Total users
total_users = df['user'].nunique()

# Total words
total_words = df['message'].str.split().str.len().sum()

print("\n========== BASIC STATISTICS ==========")
print("Total Messages :", total_messages)
print("Total Users    :", total_users)
print("Total Words    :", total_words)


# -------------------- MOST ACTIVE USERS --------------------

user_message_count = df['user'].value_counts()

print("\n========== MOST ACTIVE USERS ==========")
print(user_message_count.head(10))


# Plot top users
plt.figure(figsize=(10, 5))

user_message_count.head(10).plot(kind='bar')

plt.title('Top 10 Most Active Users')
plt.xlabel('Users')
plt.ylabel('Number of Messages')
plt.xticks(rotation=45)

plt.show()


# -------------------- MONTHLY TIMELINE --------------------

monthly_messages = df.groupby('month_year').size()

plt.figure(figsize=(12, 5))

monthly_messages.plot(
    kind='line',
    marker='o'
)

plt.title('Monthly Message Timeline')
plt.xlabel('Month')
plt.ylabel('Number of Messages')
plt.xticks(rotation=45)

plt.show()


# -------------------- DAILY TIMELINE --------------------

daily_messages = df.groupby('date').size()

plt.figure(figsize=(12, 5))

daily_messages.plot()

plt.title('Daily Message Timeline')
plt.xlabel('Date')
plt.ylabel('Number of Messages')

plt.show()


# -------------------- WEEKLY ACTIVITY --------------------

weekday_order = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday'
]

weekly_activity = (
    df['day_name']
    .value_counts()
    .reindex(weekday_order)
)

plt.figure(figsize=(10, 5))

weekly_activity.plot(kind='bar')

plt.title('Weekly Activity')
plt.xlabel('Day')
plt.ylabel('Number of Messages')
plt.xticks(rotation=45)

plt.show()


# -------------------- MONTHLY ACTIVITY --------------------

month_order = [
    'January',
    'February',
    'March',
    'April',
    'May',
    'June',
    'July',
    'August',
    'September',
    'October',
    'November',
    'December'
]

monthly_activity = (
    df['month_name']
    .value_counts()
    .reindex(month_order)
)

plt.figure(figsize=(12, 5))

monthly_activity.plot(kind='bar')

plt.title('Monthly Activity')
plt.xlabel('Month')
plt.ylabel('Number of Messages')
plt.xticks(rotation=45)

plt.show()


# -------------------- ACTIVITY HEATMAP --------------------

heatmap_data = pd.crosstab(
    df['day_name'],
    df['hour']
)

heatmap_data = heatmap_data.reindex(weekday_order)

plt.figure(figsize=(15, 6))

sns.heatmap(
    heatmap_data,
    cmap='Blues'
)

plt.title('Activity Heatmap')
plt.xlabel('Hour')
plt.ylabel('Day')

plt.show()


# -------------------- MOST COMMON WORDS --------------------

# Combine all messages
text = ' '.join(df['message'])

# Convert to lowercase and split into words
words = text.lower().split()

# Common words to remove
stop_words = {
    'the', 'is', 'a', 'to', 'and', 'of', 'in',
    'for', 'on', 'this', 'that', 'i', 'you',
    'it', 'are', 'was', 'be', 'hai', 'ho',
    'ka', 'ke', 'ki', 'ko', 'mein'
}

# Remove stop words
filtered_words = [
    word for word in words
    if word not in stop_words
]

# Count words
word_count = Counter(filtered_words)

print("\n========== MOST COMMON WORDS ==========")
print(word_count.most_common(20))


# -------------------- WORD CLOUD --------------------

clean_text = ' '.join(filtered_words)

wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color='white'
).generate(clean_text)

plt.figure(figsize=(15, 7))

plt.imshow(wordcloud)

plt.axis('off')

plt.title('Most Common Words')

plt.show()


# -------------------- EMOJI ANALYSIS --------------------

emoji_list = []

# Extract emojis from every message
for message in df['message']:

    for character in message:

        if character in emoji.EMOJI_DATA:
            emoji_list.append(character)


# Count emojis
emoji_count = Counter(emoji_list)

print("\n========== MOST USED EMOJIS ==========")
print(emoji_count.most_common(20))


# Plot top emojis
top_emojis = emoji_count.most_common(10)

emojis = [item[0] for item in top_emojis]
counts = [item[1] for item in top_emojis]

plt.figure(figsize=(10, 5))

plt.bar(emojis, counts)

plt.title('Most Used Emojis')
plt.xlabel('Emoji')
plt.ylabel('Count')

plt.show()


# -------------------- MEDIA MESSAGES --------------------

media_count = df[
    df['message'].str.contains(
        '<Media omitted>',
        case=False,
        na=False
    )
].shape[0]

print("\nTotal Media Messages:", media_count)


# -------------------- LINK ANALYSIS --------------------

url_pattern = r'https?://\S+|www\.\S+'

links = df['message'].str.findall(url_pattern)

total_links = sum(
    len(link_list)
    for link_list in links
)

print("Total Links Shared:", total_links)


# -------------------- HOURLY ACTIVITY --------------------

hourly_activity = (
    df['hour']
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(12, 5))

hourly_activity.plot(kind='bar')

plt.title('Messages by Hour')
plt.xlabel('Hour')
plt.ylabel('Number of Messages')

plt.show()


# -------------------- FINAL SUMMARY --------------------

print("\n======================================")
print("       WHATSAPP CHAT ANALYSIS")
print("======================================")

print("Total Messages      :", total_messages)
print("Total Users         :", total_users)
print("Total Words         :", total_words)
print("Total Media         :", media_count)
print("Total Links         :", total_links)

print("======================================")
print("Analysis Completed Successfully!")

FileNotFoundError: [Errno 2] No such file or directory: 'Chat.txt'

In [2]:
!pip install emoji

   ---------------------------------------- 0.0/610.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/610.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/610.9 kB ? eta -:--:--
    --------------------------------------- 10.2/610.9 kB ? eta -:--:--
   - ------------------------------------- 20.5/610.9 kB 162.5 kB/s eta 0:00:04
   -- ------------------------------------ 41.0/610.9 kB 245.8 kB/s eta 0:00:03
   ----- --------------------------------- 92.2/610.9 kB 476.3 kB/s eta 0:00:02
   --------- ---------------------------- 153.6/610.9 kB 654.6 kB/s eta 0:00:01
   --------------- ---------------------- 256.0/610.9 kB 923.9 kB/s eta 0:00:01
   ------------------------- -------------- 389.1/610.9 kB 1.2 MB/s eta 0:00:01
   -------------------------------- ------- 501.8/610.9 kB 1.4 MB/s eta 0:00:01
   -------------------------------------- - 593.9/610.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 610.9/610.9 kB 1.4 MB/s eta 0:0


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
print(os.getcwd())

C:\Users\mdnom
